In [ ]:
import os
import pandas as pd

DATA_DIR = os.getcwd()

corners = pd.read_csv(os.path.join(DATA_DIR, 'corners_data.csv'))
prices = pd.read_csv(os.path.join(DATA_DIR, 'corners_prices.csv'))
results = pd.read_csv(os.path.join(DATA_DIR, 'corners_prices_results.csv'))

display(corners.head(3))
display(prices.head(3))
display(results.head(3))

## Basic checks

In [ ]:
corners['date_time'] = pd.to_datetime(corners['date_time'], errors='coerce')
prices['date_time'] = pd.to_datetime(prices['date_time'], errors='coerce')

print('\nColumns with missing values in corners (count > 0):')
print(corners.isna().sum()[corners.isna().sum() > 0])

print('\nColumns with missing values in betting odds:')
print(prices.isna().sum()[prices.isna().sum() > 0])

print('\nColumns with missing values in match results (scores and corners):')
print(results.isna().sum()[results.isna().sum() > 0])

print('\nDuplicate match_id check (rows with same match_id):')
print('corners:', corners['match_id'].duplicated().sum(), '(should be 0)')
print('match results:', results['match_id'].duplicated().sum(), '(should be 0)')

# How many rows of oh, oa, and od are in bidding odds?
print('\nCounts of each odds_type (OU / HC / 1X2):')
print(prices['odds_type'].value_counts(dropna=False))

## Clean match result tables and the price table

In [ ]:
corners = corners.dropna(subset=['date_time', 'home_corners', 'away_corners'])

# 999 is invalid corner count
corners = corners[(corners['home_corners'] != 999) & (corners['away_corners'] != 999)]

# remove duplicate match_id with conflicting scores/corners
cols_to_check = ['home_ft_score', 'away_ft_score', 'home_corners', 'away_corners']

dup = corners[corners['match_id'].duplicated(keep=False)]
bad_corner_ids = []

for mid, g in dup.groupby('match_id'):
    if (g[cols_to_check].nunique() > 1).any():
        bad_corner_ids.append(mid)

corners = corners[~corners['match_id'].isin(bad_corner_ids)]
corners = corners.drop_duplicates()

corners['home_corners'] = corners['home_corners'].astype(int)
corners['away_corners'] = corners['away_corners'].astype(int)
corners['total_corners'] = corners['home_corners'] + corners['away_corners']
corners['corner_diff_home_minus_away'] = corners['home_corners'] - corners['away_corners']


results = results.dropna(subset=['home_corners', 'away_corners'])

# remove duplicate match_id with conflicting scores/corners
dup = results[results['match_id'].duplicated(keep=False)]
bad_result_ids = []

for mid, g in dup.groupby('match_id'):
    if (g[cols_to_check].nunique() > 1).any():
        bad_result_ids.append(mid)

results = results[~results['match_id'].isin(bad_result_ids)]
results = results.drop_duplicates()

results['home_corners'] = results['home_corners'].astype(int)
results['away_corners'] = results['away_corners'].astype(int)
results['total_corners'] = results['home_corners'] + results['away_corners']
results['corner_diff_home_minus_away'] = results['home_corners'] - results['away_corners']


display(corners.describe())
display(results.describe())


prices = prices.drop_duplicates()
# drop rows where date_time could not be parsed (14 rows) — needed for Q2 time-based plots
prices = prices.dropna(subset=['date_time'])

prices_missing_odds = prices[prices[['oh', 'oa', 'od']].isna().any(axis=1)]
prices_with_odds = prices.dropna(subset=['oh', 'oa', 'od'])

# oh and oa must be > 1
prices_with_odds = prices_with_odds[
    (prices_with_odds['oh'] > 1) &
    (prices_with_odds['oa'] > 1) &
    (prices_with_odds['odds_type'].isin(['OU', 'HC', '1X2']))
]

# For OU: od is the line (must be > 0); for 1X2: od is the draw price (must be > 1 as decimal odds)
prices_with_odds = prices_with_odds[
    ((prices_with_odds['odds_type'] == 'OU') & (prices_with_odds['od'] > 0)) |
    ((prices_with_odds['odds_type'] == '1X2') & (prices_with_odds['od'] > 1)) |
    ((prices_with_odds['odds_type'] == 'HC'))
]


## Build the Q2 betting evaluation set

Use an inner join because Q2 needs both prices and realised corner results.

In [ ]:
betting = prices_with_odds.merge(
    results[['match_id', 'home_ft_score', 'away_ft_score', 'home_corners', 'away_corners', 'total_corners', 'corner_diff_home_minus_away']],
    on='match_id',
    how='inner',
    suffixes=('', '_result')
)

display(betting.head(3))

# Compare the distribution of corners in the betting dataset vs corners_data
# As one will be used for training and the other for testing, we want to check if they have similar distributions 
betting_matches = betting.drop_duplicates('match_id')

# Compare corner count summary
corner_cols = ['home_corners', 'away_corners', 'total_corners']

summary = pd.concat([
    corners[corner_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
        .add_prefix('corners_data_'),
    betting_matches[corner_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
        .add_prefix('betting_')
], axis=1)

display(summary)

## Train vs betting distribution check

This decides whether `corners_data.csv` is representative enough for the later betting task.

In [ ]:
# Visual comparison of corner distributions
import matplotlib.pyplot as plt

for col in ['home_corners', 'away_corners', 'total_corners']:
    plt.figure(figsize=(7, 4))
    plt.hist(corners[col], bins=25, density=True, alpha=0.6, label='corners_data')
    plt.hist(betting_matches[col], bins=25, density=True, alpha=0.6, label='betting_matches')
    plt.xlabel(col)
    
    plt.ylabel('Density')
    plt.title(f'Distribution of {col}')
    plt.legend()
    plt.show()


# Competition distribution
competition_dist = pd.concat([
    corners['competition_id'].value_counts(normalize=True).rename('corners_data'),
    betting_matches['competition_id'].value_counts(normalize=True).rename('betting_matches')
], axis=1).fillna(0).sort_index()

display(competition_dist)

competition_dist.plot(kind='bar', figsize=(8, 4))
plt.ylabel('Share of matches')
plt.title('Competition distribution')
plt.show()

# Season distribution
season_dist = pd.concat([
    corners['season_id'].value_counts(normalize=True).rename('corners_data'),
    betting_matches['season_id'].value_counts(normalize=True).rename('betting_matches')
], axis=1).fillna(0).sort_index()

display(season_dist)


# Importance Weighting

Reweight train by competition share to match betting distribution, 
then re-visualise corner densities.

In [ ]:
import numpy as np
import pandas as pd

# =========================
# Train data
# =========================
train = corners.copy()

# =========================
# Compute competition distributions
# =========================
train_comp = train["competition_id"].value_counts(normalize=True)
betting_comp = betting_matches["competition_id"].value_counts(normalize=True)

# =========================
# Importance weights
# =========================
def get_weight(c):
    return betting_comp.get(c, 0) / train_comp.get(c, 1e-9)

train["iw"] = train["competition_id"].map(get_weight)
train["iw"] = train["iw"] / train["iw"].mean()

# =========================
# Mean comparison
# =========================
cols = ["home_corners", "away_corners", "total_corners"]

for col in cols:
    mask = train[col].notna()

    original_mean   = train[col].mean()
    reweighted_mean = np.average(train.loc[mask, col], weights=train.loc[mask, "iw"])
    betting_mean    = betting_matches[col].mean()

    print(f"\n{col}")
    print("original mean:", round(original_mean, 4))
    print("reweighted mean:", round(reweighted_mean, 4))
    print("betting mean    :", round(betting_mean, 4))

In [ ]:

corners["match_id"].duplicated().sum()
print("duplicates in corners:", corners["match_id"].duplicated().sum())

dupes = corners[corners.duplicated(subset="match_id", keep=False)]
dupes = dupes.sort_values("match_id")
print("Number of duplicated rows:", len(dupes))
display(dupes)

overlap = set(corners["match_id"]) & set(results["match_id"])
print("overlap:", len(overlap))

In [ ]:
for col in ['home_corners', 'away_corners']:
    m, v = corners[col].mean(), corners[col].var()
    print(f"{col}: mean={m:.2f}, var={v:.2f}, ratio={v/m:.2f}")

# train: corners_data with importance weights — used for model training (Q1)
# betting: prices + realised results — used for Q2 EV/PnL evaluation
train.to_parquet('train.parquet', index=False)
betting.to_parquet('betting.parquet', index=False)
print("Saved train.parquet:", train.shape, "| betting.parquet:", betting.shape)
